# Setup

In [1]:
# =========================================
# SETUP (Part 1 only)
# =========================================
# Yêu cầu: không tạo thêm cây thư mục (logs/outputs/src/notebooks/...),
# chỉ sử dụng thư mục dữ liệu có sẵn.

from pathlib import Path
import numpy as np
import pandas as pd

DATA_RAW = Path('datathon2026') / 'data' / 'raw'

REQUIRED_FILES_PART1 = [
    'orders.csv',
    'order_items.csv',
    'products.csv',
    'customers.csv',
    'geography.csv',
    'returns.csv',
    'web_traffic.csv',
    'payments.csv',
]

OPTIONAL_FILES = [
    'sales_train.csv',  # nếu BTC có cung cấp theo đề
]


def load_all_data_part1(data_dir: Path = DATA_RAW) -> dict[str, pd.DataFrame]:
    # Không copy/symlink, không mkdir.
    if not data_dir.exists():
        raise FileNotFoundError(f'Data directory not found: {data_dir.resolve()}')

    missing = [f for f in REQUIRED_FILES_PART1 if not (data_dir / f).exists()]
    if missing:
        raise FileNotFoundError(f'Missing required CSV(s) in {data_dir}: {missing}')

    data: dict[str, pd.DataFrame] = {}

    for fname in REQUIRED_FILES_PART1:
        fp = data_dir / fname
        data[fp.stem] = pd.read_csv(fp, low_memory=False)

    for fname in OPTIONAL_FILES:
        fp = data_dir / fname
        if fp.exists():
            data[fp.stem] = pd.read_csv(fp, low_memory=False)

    print('Loaded Part 1 tables:')
    for k in sorted(data.keys()):
        print(f"- {k}: {data[k].shape}")

    return data


data = load_all_data_part1()


Loaded Part 1 tables:
- customers: (121930, 7)
- geography: (39948, 4)
- order_items: (714669, 7)
- orders: (646945, 8)
- payments: (646945, 4)
- products: (2412, 8)
- returns: (39939, 7)
- web_traffic: (3652, 7)


# Part 1

In [2]:
# === Q1: Inter-order gap median (days) ===
"""
Tính trung vị số ngày giữa 2 lần mua liên tiếp (inter-order gap).
- Dùng orders[order_date, customer_id]
- Sort theo thời gian trong từng khách
- Lấy diff() theo ngày, chỉ giữ khách có >1 đơn (tức có gap)
- Median trên toàn bộ các gap và map về option gần nhất: 30/90/144/365
"""
import numpy as np
import pandas as pd

# --- Load & sanity input ---
data = globals().get("data") or globals().get("all_data")
assert data is not None, "Chưa có dict `data` (hoặc `all_data`). Hãy chạy load_all_data_part1() trước."
orders = data["orders"].copy()
required = {"order_date", "customer_id"}
assert required.issubset(orders.columns), f"orders thiếu cột: {required - set(orders.columns)}"

# --- Parse datetime ---
orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
orders = orders.dropna(subset=["order_date", "customer_id"])

# --- Compute gaps within each customer ---
orders = orders.sort_values(["customer_id", "order_date"])
gaps = orders.groupby("customer_id")["order_date"].diff()
gap_days = gaps.dt.days.dropna()

# --- Keep only customers with >1 order is implicitly satisfied by diff() non-null ---
assert len(gap_days) > 0, "Không có inter-order gap nào để tính (có thể mọi khách chỉ có 1 đơn)."

median_gap = float(np.median(gap_days.values))

# --- Map to options ---
def pick_nearest_option(value: float, choices: dict[str, float]) -> str:
    """Pick the nearest option key by absolute distance.

    Args:
        value (float): Computed numeric value.
        choices (dict[str, float]): Mapping option letter -> numeric target.

    Returns:
        str: Option letter (e.g., 'A').

    Raises:
        ValueError: If choices is empty.
    """
    if not choices:
        raise ValueError("choices must be non-empty")
    return min(choices, key=lambda k: abs(value - choices[k]))

options = {"A": 30.0, "B": 90.0, "C": 144.0, "D": 365.0}
option = pick_nearest_option(median_gap, options)

result = round(median_gap, 2)
print(f"Q1 → {result} days → Đáp án: {option}")

# --- Sanity checks ---
assert option in {"A", "B", "C", "D"}
assert median_gap >= 0


Q1 → 144.0 days → Đáp án: C


In [3]:
# === Q2: Segment with highest average gross margin ===
"""
Tìm segment có gross margin trung bình cao nhất.
- Gross margin = (price - cogs) / price
- Groupby segment, lấy mean gross_margin rồi idxmax
- Map segment về option: Premium/Performance/Activewear/Standard
"""
import numpy as np
import pandas as pd

data = globals().get("data") or globals().get("all_data")
assert data is not None, "Chưa có dict `data` (hoặc `all_data`). Hãy chạy load_all_data_part1() trước."
products = data["products"].copy()
required = {"segment", "price", "cogs"}
assert required.issubset(products.columns), f"products thiếu cột: {required - set(products.columns)}"

# --- Compute gross margin safely ---
products["price"] = pd.to_numeric(products["price"], errors="coerce")
products["cogs"] = pd.to_numeric(products["cogs"], errors="coerce")
products = products.dropna(subset=["segment", "price", "cogs"])
products = products[products["price"] != 0]

products["gross_margin"] = (products["price"] - products["cogs"]) / products["price"]

gm_by_seg = products.groupby("segment", dropna=True)["gross_margin"].mean().sort_values(ascending=False)
assert len(gm_by_seg) > 0, "Không tính được gross margin theo segment."

best_segment = gm_by_seg.index[0]
result = str(best_segment)

# --- Map to options ---
seg_to_option = {"Premium": "A", "Performance": "B", "Activewear": "C", "Standard": "D"}
option = seg_to_option.get(best_segment, "N/A")

print(f"Q2 → {result} → Đáp án: {option}")

# --- Sanity checks ---
assert option in {"A", "B", "C", "D", "N/A"}
assert np.isfinite(gm_by_seg.iloc[0])

Q2 → Standard → Đáp án: D


In [4]:
# === Q3: Most common return reason in Streetwear ===
"""
Tìm lý do trả hàng phổ biến nhất trong category Streetwear.
- Join returns với products theo product_id để lấy category
- Filter category == 'Streetwear'
- value_counts trên return_reason lấy top-1
- Map về option: defective/wrong_size/changed_mind/not_as_described
"""
import pandas as pd

assert "data" in globals(), "Biến `data` chưa tồn tại. Hãy chạy load_all_data_part1() trước."
returns = data["returns"].copy()
products = data["products"].copy()

required_r = {"product_id", "return_reason"}
required_p = {"product_id", "category"}
assert required_r.issubset(returns.columns), f"returns thiếu cột: {required_r - set(returns.columns)}"
assert required_p.issubset(products.columns), f"products thiếu cột: {required_p - set(products.columns)}"

# --- Join & filter ---
rp = returns.merge(products[["product_id", "category"]], on="product_id", how="left")
street = rp[rp["category"] == "Streetwear"].copy()
assert len(street) > 0, "Không có bản ghi returns thuộc Streetwear."

# --- Top reason ---
vc = street["return_reason"].astype(str).value_counts(dropna=False)
top_reason = vc.index[0]
result = str(top_reason)

reason_to_option = {
    "defective": "A",
    "wrong_size": "B",
    "changed_mind": "C",
    "not_as_described": "D",
}
option = reason_to_option.get(top_reason, "N/A")

print(f"Q3 → {result} → Đáp án: {option}")

# --- Sanity checks ---
assert option in {"A", "B", "C", "D", "N/A"}
assert vc.iloc[0] >= 1

Q3 → wrong_size → Đáp án: B


In [5]:
# === Q4: Traffic source with lowest average bounce_rate ===
"""
Tìm traffic_source có bounce_rate trung bình thấp nhất.
- Groupby traffic_source, mean(bounce_rate), lấy idxmin
- Map về option: organic_search/paid_search/email_campaign/social_media
"""
import numpy as np
import pandas as pd

assert "data" in globals(), "Biến `data` chưa tồn tại. Hãy chạy load_all_data_part1() trước."
wt = data["web_traffic"].copy()

required = {"traffic_source", "bounce_rate"}
assert required.issubset(wt.columns), f"web_traffic thiếu cột: {required - set(wt.columns)}"

# --- Clean types ---
wt["bounce_rate"] = pd.to_numeric(wt["bounce_rate"], errors="coerce")
wt = wt.dropna(subset=["traffic_source", "bounce_rate"])

avg_bounce = wt.groupby("traffic_source")["bounce_rate"].mean().sort_values()
assert len(avg_bounce) > 0, "Không tính được bounce_rate trung bình theo traffic_source."

best_source = avg_bounce.index[0]
result = str(best_source)

src_to_option = {
    "organic_search": "A",
    "paid_search": "B",
    "email_campaign": "C",
    "social_media": "D",
}
option = src_to_option.get(best_source, "N/A")

print(f"Q4 → {result} → Đáp án: {option}")

# --- Sanity checks ---
assert option in {"A", "B", "C", "D", "N/A"}
assert np.isfinite(avg_bounce.iloc[0])

Q4 → email_campaign → Đáp án: C


In [6]:
# === Q5: % order_items rows with non-null promo_id ===
"""
Tính % dòng order_items có promo_id không null.
- pct = promo_id.notna().sum() / len(order_items) * 100
- Map về option gần nhất: 12/25/39/54 (%)
"""
import numpy as np
import pandas as pd

assert "data" in globals(), "Biến `data` chưa tồn tại. Hãy chạy load_all_data_part1() trước."
oi = data["order_items"].copy()

required = {"promo_id"}
assert required.issubset(oi.columns), f"order_items thiếu cột: {required - set(oi.columns)}"
assert len(oi) > 0, "order_items rỗng."

# --- Compute percentage ---
non_null = int(oi["promo_id"].notna().sum())
pct = non_null / len(oi) * 100.0

def pick_nearest_option(value: float, choices: dict[str, float]) -> str:
    """Pick the nearest option key by absolute distance.

    Args:
        value (float): Computed numeric value.
        choices (dict[str, float]): Mapping option letter -> numeric target.

    Returns:
        str: Option letter (e.g., 'A').

    Raises:
        ValueError: If choices is empty.
    """
    if not choices:
        raise ValueError("choices must be non-empty")
    return min(choices, key=lambda k: abs(value - choices[k]))

options = {"A": 12.0, "B": 25.0, "C": 39.0, "D": 54.0}
option = pick_nearest_option(pct, options)

result = f"{pct:.2f}%"
print(f"Q5 → {result} → Đáp án: {option}")

# --- Sanity checks ---
assert 0.0 <= pct <= 100.0
assert option in {"A", "B", "C", "D"}

Q5 → 38.66% → Đáp án: C


In [7]:
# === Q6: Age group with highest avg orders per customer ===
"""
Tìm age_group có số đơn trung bình/khách cao nhất.
- Lọc customers có age_group không null
- Đếm số đơn theo customer_id từ orders
- Merge vào customers rồi groupby age_group -> mean(order_count) -> idxmax
- Map về option: 55+ / 25-34 / 35-44 / 45-54
"""
import pandas as pd

assert "data" in globals(), "Biến `data` chưa tồn tại. Hãy chạy load_all_data_part1() trước."
customers = data["customers"].copy()
orders = data["orders"].copy()

req_c = {"customer_id", "age_group"}
req_o = {"customer_id", "order_id"}
assert req_c.issubset(customers.columns), f"customers thiếu cột: {req_c - set(customers.columns)}"
assert req_o.issubset(orders.columns), f"orders thiếu cột: {req_o - set(orders.columns)}"

# --- Filter valid age groups ---
cust = customers.dropna(subset=["customer_id", "age_group"]).copy()
assert len(cust) > 0, "Không có customer có age_group hợp lệ."

# --- Orders per customer ---
order_cnt = orders.dropna(subset=["customer_id", "order_id"]).groupby("customer_id")["order_id"].nunique()
cust = cust.merge(order_cnt.rename("order_count"), on="customer_id", how="left")
cust["order_count"] = cust["order_count"].fillna(0)

avg_by_age = cust.groupby("age_group")["order_count"].mean().sort_values(ascending=False)
assert len(avg_by_age) > 0, "Không tính được average orders theo age_group."

best_age = avg_by_age.index[0]
result = str(best_age)

age_to_option = {"55+": "A", "25-34": "B", "35-44": "C", "45-54": "D"}
option = age_to_option.get(best_age, "N/A")

print(f"Q6 → {result} → Đáp án: {option}")

# --- Sanity checks ---
assert option in {"A", "B", "C", "D", "N/A"}
assert avg_by_age.iloc[0] >= 0

Q6 → 55+ → Đáp án: A


In [8]:
# === Q7: Region with highest total revenue ===
# Theo đề: geography.csv + sales_train.csv
# A) West | B) Central | C) East | D) Cả ba vùng xấp xỉ bằng nhau
#
# Triển khai:
# - Nếu có sales_train.csv: ưu tiên dùng (flex schema: region+revenue, hoặc order_id/zip + revenue).
# - Nếu không có sales_train.csv trong data hiện tại: fallback tính revenue từ order_items
#   (quantity * unit_price - discount_amount) rồi join orders(zip) -> geography(region).

import numpy as np
import pandas as pd

assert 'data' in globals(), "Biến `data` chưa tồn tại. Hãy chạy load_all_data_part1() trước."

def _region_revenue_from_order_items(data: dict[str, pd.DataFrame]) -> pd.Series:
    orders = data['orders'].copy()
    geo = data['geography'].copy()
    oi = data['order_items'].copy()

    req_o = {'order_id', 'zip'}
    req_g = {'zip', 'region'}
    req_oi = {'order_id', 'quantity', 'unit_price', 'discount_amount'}
    assert req_o.issubset(orders.columns), f"orders thiếu cột: {req_o - set(orders.columns)}"
    assert req_g.issubset(geo.columns), f"geography thiếu cột: {req_g - set(geo.columns)}"
    assert req_oi.issubset(oi.columns), f"order_items thiếu cột: {req_oi - set(oi.columns)}"

    oi['quantity'] = pd.to_numeric(oi['quantity'], errors='coerce')
    oi['unit_price'] = pd.to_numeric(oi['unit_price'], errors='coerce')
    oi['discount_amount'] = pd.to_numeric(oi['discount_amount'], errors='coerce')
    oi = oi.dropna(subset=['order_id', 'quantity', 'unit_price', 'discount_amount'])

    oi['revenue_line'] = oi['quantity'] * oi['unit_price'] - oi['discount_amount']

    ord_geo = orders[['order_id', 'zip']].merge(geo[['zip', 'region']], on='zip', how='left')
    merged = oi[['order_id', 'revenue_line']].merge(ord_geo, on='order_id', how='left')
    merged = merged.dropna(subset=['region'])

    return merged.groupby('region')['revenue_line'].sum().sort_values(ascending=False)


def _region_revenue_from_sales_train(data: dict[str, pd.DataFrame]):
    if 'sales_train' not in data:
        return None

    st = data['sales_train'].copy()
    cols = set(st.columns)

    # Find revenue-like column
    revenue_col = next((c for c in st.columns if c.lower() in {'revenue','sales','total_revenue','gmv'}), None)
    region_col = next((c for c in st.columns if c.lower() == 'region'), None)

    if revenue_col is None:
        return None

    st[revenue_col] = pd.to_numeric(st[revenue_col], errors='coerce')

    # Case 1: region + revenue
    if region_col is not None:
        tmp = st.dropna(subset=[region_col, revenue_col])
        if len(tmp) == 0:
            return None
        return tmp.groupby(region_col)[revenue_col].sum().sort_values(ascending=False)

    # Case 2: order_id + revenue -> join orders->geo
    if 'order_id' in cols:
        orders = data['orders'][['order_id', 'zip']].copy()
        geo = data['geography'][['zip', 'region']].copy()
        tmp = st.dropna(subset=['order_id', revenue_col]).merge(orders, on='order_id', how='left').merge(geo, on='zip', how='left')
        tmp = tmp.dropna(subset=['region'])
        if len(tmp) == 0:
            return None
        return tmp.groupby('region')[revenue_col].sum().sort_values(ascending=False)

    # Case 3: zip + revenue -> join geo
    if 'zip' in cols:
        geo = data['geography'][['zip', 'region']].copy()
        tmp = st.dropna(subset=['zip', revenue_col]).merge(geo, on='zip', how='left').dropna(subset=['region'])
        if len(tmp) == 0:
            return None
        return tmp.groupby('region')[revenue_col].sum().sort_values(ascending=False)

    return None


rev_by_region = _region_revenue_from_sales_train(data)
used = 'sales_train.csv' if rev_by_region is not None else 'order_items fallback'
if rev_by_region is None:
    rev_by_region = _region_revenue_from_order_items(data)

assert len(rev_by_region) > 0, 'Không tính được revenue theo region.'

max_rev = float(rev_by_region.iloc[0])
min_rev = float(rev_by_region.iloc[-1])
approx_equal = (max_rev > 0) and ((max_rev - min_rev) / max_rev < 0.01)

best_region = str(rev_by_region.index[0])
region_to_option = {'West': 'A', 'Central': 'B', 'East': 'C'}
option = 'D' if approx_equal else region_to_option.get(best_region, 'N/A')

print(f"Q7 → {best_region} (source={used}) → Đáp án: {option}")

assert option in {'A','B','C','D','N/A'}
assert np.isfinite(max_rev)


Q7 → East (source=order_items fallback) → Đáp án: C


In [9]:
# === Q8: Most common payment method among cancelled orders ===
"""
Tìm payment_method phổ biến nhất trong các đơn bị huỷ.
- Filter orders[order_status == 'cancelled']
- value_counts(payment_method) lấy top-1
- Map về option: credit_card/cod/paypal/bank_transfer
"""
import pandas as pd

assert "data" in globals(), "Biến `data` chưa tồn tại. Hãy chạy load_all_data_part1() trước."
orders = data["orders"].copy()

req = {"order_status", "payment_method"}
assert req.issubset(orders.columns), f"orders thiếu cột: {req - set(orders.columns)}"

cancelled = orders[orders["order_status"] == "cancelled"].copy()
assert len(cancelled) > 0, "Không có đơn bị huỷ (order_status == 'cancelled')."

vc = cancelled["payment_method"].astype(str).value_counts()
top_method = vc.index[0]
result = str(top_method)

method_to_option = {"credit_card": "A", "cod": "B", "paypal": "C", "bank_transfer": "D"}
option = method_to_option.get(top_method, "N/A")

print(f"Q8 → {result} → Đáp án: {option}")

# --- Sanity checks ---
assert option in {"A", "B", "C", "D", "N/A"}
assert vc.iloc[0] >= 1

Q8 → credit_card → Đáp án: A


In [10]:
# === Q9: Size with highest return rate ===
"""
Tìm size có tỷ lệ trả hàng cao nhất.
- Numerator: returns + products (product_id) -> count returns theo size
- Denominator: order_items + products (product_id) -> count order_items rows theo size
- Rate = returns_count / order_items_count
- Chọn size có rate cao nhất và map option: S/M/L/XL
"""
import numpy as np
import pandas as pd

assert "data" in globals(), "Biến `data` chưa tồn tại. Hãy chạy load_all_data_part1() trước."
returns = data["returns"].copy()
oi = data["order_items"].copy()
products = data["products"].copy()

req_r = {"product_id"}
req_oi = {"product_id"}
req_p = {"product_id", "size"}
assert req_r.issubset(returns.columns), f"returns thiếu cột: {req_r - set(returns.columns)}"
assert req_oi.issubset(oi.columns), f"order_items thiếu cột: {req_oi - set(oi.columns)}"
assert req_p.issubset(products.columns), f"products thiếu cột: {req_p - set(products.columns)}"

# --- Counts by size (returns) ---
ret_sz = returns.merge(products[["product_id", "size"]], on="product_id", how="left")
ret_sz = ret_sz.dropna(subset=["size"])
returns_count = ret_sz.groupby("size").size().rename("returns_count")

# --- Counts by size (order_items lines) ---
oi_sz = oi.merge(products[["product_id", "size"]], on="product_id", how="left")
oi_sz = oi_sz.dropna(subset=["size"])
order_items_count = oi_sz.groupby("size").size().rename("order_items_count")

# --- Compute rates ---
rates = pd.concat([returns_count, order_items_count], axis=1).fillna(0)
rates = rates[rates["order_items_count"] > 0].copy()
rates["return_rate"] = rates["returns_count"] / rates["order_items_count"]

# Focus on options sizes only
rates = rates.loc[rates.index.astype(str).isin(["S", "M", "L", "XL"])]
assert len(rates) > 0, "Không đủ dữ liệu để tính return_rate cho size trong {S,M,L,XL}."

best_size = rates["return_rate"].idxmax()
best_rate = float(rates.loc[best_size, "return_rate"])
result = f"{best_size} (rate={best_rate:.4f})"

size_to_option = {"S": "A", "M": "B", "L": "C", "XL": "D"}
option = size_to_option.get(str(best_size), "N/A")

print(f"Q9 → {result} → Đáp án: {option}")

# --- Sanity checks ---
assert option in {"A", "B", "C", "D", "N/A"}
assert 0.0 <= best_rate
assert np.isfinite(best_rate)

Q9 → S (rate=0.0565) → Đáp án: A


In [11]:
# === Q10: Installments with highest avg payment_value per order (1,3,6,12) ===
# Theo đề: payments.csv
# A) 1 kỳ | B) 3 kỳ | C) 6 kỳ | D) 12 kỳ
#
# Lấy payment_value trung bình *trên mỗi đơn hàng* cho từng installments trong {1,3,6,12}.

import numpy as np
import pandas as pd

assert 'data' in globals(), "Biến `data` chưa tồn tại. Hãy chạy load_all_data_part1() trước."
pay = data['payments'].copy()

req = {'order_id', 'installments', 'payment_value'}
assert req.issubset(pay.columns), f"payments thiếu cột: {req - set(pay.columns)}"

pay['installments'] = pd.to_numeric(pay['installments'], errors='coerce')
pay['payment_value'] = pd.to_numeric(pay['payment_value'], errors='coerce')
pay = pay.dropna(subset=['order_id', 'installments', 'payment_value'])

allowed = {1, 3, 6, 12}

# Per-order amount (sum in case future data has multiple payment records/order)
per_order = (
    pay.assign(installments=pay['installments'].astype(int))
       .query('installments in @allowed')
       .groupby(['order_id', 'installments'], as_index=False)['payment_value'].sum()
)
assert len(per_order) > 0, 'Không có bản ghi payments với installments trong {1,3,6,12}.'

avg_pay = per_order.groupby('installments')['payment_value'].mean().sort_values(ascending=False)
best_inst = int(avg_pay.index[0])

inst_to_option = {1: 'A', 3: 'B', 6: 'C', 12: 'D'}
option = inst_to_option.get(best_inst, 'N/A')

print(f"Q10 → {best_inst} installments → Đáp án: {option}")

assert option in {'A','B','C','D','N/A'}
assert np.isfinite(avg_pay.iloc[0])


Q10 → 6 installments → Đáp án: C
